# 🛡️ DeepShield — Training on Google Colab
**Step-by-step guide to train XceptionNet + EfficientNet-B4 on FaceForensics++**

Runtime → Change runtime type → **T4 GPU** before running!

In [ ]:
# ── Cell 1: Check GPU ──
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE ❌')
print('CUDA:', torch.version.cuda)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ── Cell 2: Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')
# Your checkpoints will be saved to Drive so they persist after Colab disconnects

In [ ]:
# ── Cell 3: Install dependencies ──
!pip install timm tqdm scikit-learn matplotlib -q
print('✅ Dependencies installed')

In [ ]:
# ── Cell 4: Upload DeepShield project files ──
# Option A: Upload zip from your computer
from google.colab import files
uploaded = files.upload()  # Upload deepshield.zip

import zipfile, os
with zipfile.ZipFile('deepshield.zip', 'r') as z:
    z.extractall('/content/')
os.chdir('/content/deepshield')
print('✅ Project extracted')
!ls

In [ ]:
# ── Cell 5A: Option A — Use SYNTHETIC data (test pipeline, no real dataset needed) ──
# This lets you verify training works before getting FF++ dataset
# WARNING: synthetic data will NOT produce a real deepfake detector

!python prepare_dataset.py --mode synthetic --output ./data --n_samples 500
!echo 'Dataset ready:' && find ./data -name '*.jpg' | wc -l && echo 'images'

In [ ]:
# ── Cell 5B: Option B — FaceForensics++ (REAL dataset, best results) ──
# 1. Request access at: https://github.com/ondyari/FaceForensics
# 2. Download: original_sequences + manipulated_sequences (Deepfakes + Face2Face)
# 3. Upload to Google Drive at: /content/drive/MyDrive/ff++/
# Then run:

# !python prepare_dataset.py \
#     --mode video \
#     --source /content/drive/MyDrive/ff++ \
#     --output ./data \
#     --max_frames 30
print('Uncomment above lines after uploading FF++ dataset to Drive')

In [ ]:
# ── Cell 6: Train XceptionNet (main model) ──
!python train.py \
    --model xception \
    --data_dir ./data \
    --output_dir ./checkpoints \
    --log_dir ./training_logs \
    --epochs 20 \
    --batch_size 32 \
    --lr 1e-4 \
    --pretrained \
    --fp16 \
    --colab

In [ ]:
# ── Cell 7: Train EfficientNet-B4 (secondary model) ──
!python train.py \
    --model efficientnet_b4 \
    --data_dir ./data \
    --output_dir ./checkpoints \
    --log_dir ./training_logs \
    --epochs 20 \
    --batch_size 24 \
    --lr 1e-4 \
    --pretrained \
    --fp16 \
    --colab

In [ ]:
# ── Cell 8: View training plots ──
from IPython.display import Image as IPImage, display
import os

for plot in ['xception_progress.png', 'xception_final_eval.png',
             'efficientnet_b4_progress.png', 'efficientnet_b4_final_eval.png']:
    path = f'./training_logs/{plot}'
    if os.path.exists(path):
        print(f'\n--- {plot} ---')
        display(IPImage(path))

In [ ]:
# ── Cell 9: Save checkpoints to Google Drive ──
import shutil, os

drive_out = '/content/drive/MyDrive/deepshield_checkpoints'
os.makedirs(drive_out, exist_ok=True)

for f in os.listdir('./checkpoints'):
    if f.endswith('.pth'):
        shutil.copy(f'./checkpoints/{f}', f'{drive_out}/{f}')
        print(f'✅ Saved: {f}')

print(f'\nCheckpoints saved to Google Drive: {drive_out}')
print('Download them and put into your local deepshield/checkpoints/ folder')

In [ ]:
# ── Cell 10: Download checkpoints directly ──
from google.colab import files
import os

for f in ['xception_deepfake.pth', 'efficientnet_b4_deepfake.pth']:
    path = f'./checkpoints/{f}'
    if os.path.exists(path):
        files.download(path)
        print(f'⬇️  Downloading {f}...')
    else:
        print(f'⚠️  {f} not found')

## ✅ After downloading checkpoints

1. Place the `.pth` files in your local `deepshield/checkpoints/` folder:
```
deepshield/
  checkpoints/
    xception_deepfake.pth
    efficientnet_b4_deepfake.pth
```
2. Restart the app: `streamlit run app.py`
3. The sidebar will show **✅ Trained model** instead of the demo warning
4. All predictions will now be real!

## Expected accuracy on FaceForensics++
| Epochs | Accuracy | AUC |
|--------|----------|-----|
| 5      | ~85%     | ~0.91 |
| 10     | ~91%     | ~0.96 |
| 20     | ~94%     | ~0.98 |